# Character-Level Language Model Practice

**Topic:** NLP and Generative AI Review &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `corpus`: one long training string (~60,000 characters, alphabet `a`–`j`).
- `test_contexts`: 2,500 strings of 6 characters each. Predict the character that follows each. `y_test` (the true next characters) is used by the evaluation cell.
- Hidden rule: with probability 0.78 the next character is a single fixed preferred character for that two-character context, and otherwise it is uniform over the alphabet, so the best possible next-character accuracy is about 0.80.

## Your task

1. Build next-character counts from `corpus` conditioned on the previous two characters (a dict or a 10×10×10 array).
2. Add a backoff: if a two-character context was never seen, fall back to one-character (bigram) counts.
3. For each context in `test_contexts`, predict the most likely next character and set `predictions` to the list/array of predicted characters.

## Evaluation

Next-character accuracy must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np

rng = np.random.default_rng(511)
ALPHABET = "abcdefghij"
K = len(ALPHABET)

# hidden order-2 transition rule: each (c1, c2) context strongly prefers one next char
_pref = rng.integers(0, K, (K, K))

def _next_char(c1, c2, r):
    if r.random() < 0.78:
        return _pref[c1, c2]
    return r.integers(0, K)

def _gen(length, seed):
    r = np.random.default_rng(seed)
    seq = [r.integers(0, K), r.integers(0, K)]
    for _ in range(length - 2):
        seq.append(_next_char(seq[-2], seq[-1], r))
    return "".join(ALPHABET[i] for i in seq)

corpus = _gen(60_000, 521)

_test_raw = _gen(17_500, 522)
_starts = list(range(0, 2_500 * 7, 7))
test_contexts = [_test_raw[i:i+6] for i in _starts]
y_test = np.array([_test_raw[i+6] for i in _starts])

print(f"corpus sample: {corpus[:60]}...")
print(f"{len(test_contexts)} test contexts, e.g. {test_contexts[0]!r} -> ?")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: next-character accuracy
# =====================================================================
import numpy as np

TARGET = 0.65

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.array([str(p) for p in predictions])
if _pred.shape != y_test.shape:
    raise ValueError(f"`predictions` has length {len(_pred)}, expected {len(y_test)}.")

accuracy = float((_pred == y_test).mean())
print("=" * 52)
print(f"Next-char accuracy : {accuracy:.4f}   (target: >= {TARGET}, ceiling ~0.80)")
print(f"Chance level       : {1/10:.2f}")
print("=" * 52)
print("PASSED: you built a language model!" if accuracy >= TARGET else "NOT PASSED yet: condition on TWO characters, with backoff.")


## Solution

Count next-character frequencies conditioned on the previous two characters in a 10x10x10 array, with a 10x10 bigram table as backoff for unseen contexts. Prediction is an argmax lookup on the last two characters of each context.

```python
import numpy as np

K = len(ALPHABET)
idx = {ch: i for i, ch in enumerate(ALPHABET)}

tri = np.zeros((K, K, K), dtype=int)
bi = np.zeros((K, K), dtype=int)
for a, b, c in zip(corpus, corpus[1:], corpus[2:]):
    tri[idx[a], idx[b], idx[c]] += 1
for b, c in zip(corpus, corpus[1:]):
    bi[idx[b], idx[c]] += 1

predictions = []
for ctx in test_contexts:
    c1, c2 = idx[ctx[-2]], idx[ctx[-1]]
    counts = tri[c1, c2]
    if counts.sum() == 0:
        counts = bi[c2]                      # backoff to bigram
    predictions.append(ALPHABET[int(counts.argmax())])
```
